# Introduction to Deep Learning '26 Assignment A0

Paul Tielens, s3612031

In [3]:
import torch
import numpy as np

print(torch.cuda.is_available()) 
print(torch.cuda.device_count()) 
# print(torch.cuda.get_device_name(0))

False
0


In [5]:
# constants

nr_digits = 10

data_folder = "data/"

In [6]:
# Load data
train_in = np.loadtxt(data_folder + "train_in.csv", delimiter=",")
train_out = np.loadtxt(data_folder + "train_out.csv", delimiter=",")

test_out = np.loadtxt(data_folder + "test_out.csv", delimiter=",")
test_in = np.loadtxt(data_folder + "test_in.csv", delimiter=",")


In [39]:
values, counts = np.unique(test_out, return_counts=True)
for value, count in zip(values, counts):
    print(f"{int(value)}: {count}")

# test data is a bit unbalanced
# train data probably also

0: 224
1: 121
2: 101
3: 79
4: 86
5: 55
6: 90
7: 64
8: 92
9: 88


# Task 1.1

We would expect that digits with similar average vectors would be the most difficult to separate

In [22]:
# Calculate average vector for every digit from the training data

avg_vectors = []

for i in range(nr_digits):
    indices = np.where(train_out == i)

    vectors_i = train_in[indices]
    avg_vector_i = np.mean(vectors_i, axis=0)

    avg_vectors.append(avg_vector_i)

avg_vectors = np.array(avg_vectors)

In [40]:
# Classify the digits based on the distance between the average vector and the test vector

predictions = []

for test_in_element in test_in:
    distances = []
    for i in range(nr_digits):
        dist = np.linalg.norm(avg_vectors[i] - test_in_element)
        distances.append(dist)

    # Take the index of the digit with the lowest 'distance'
    predictions.append(np.argmin(distances))

vector_avg_accuracy = (test_out == predictions).mean() 

print(vector_avg_accuracy)

0.804


In [7]:
lr = 0.1

b = np.ones(len(train_in))
T = np.column_stack((train_in, b)) # Append column of ones to the input
W_random = np.random.randn(257, 10)
W_zeros = np.zeros((257, 10))
W_ones = np.ones((257, 10))

b_test = np.ones(len(test_in))
T_test = np.column_stack((test_in, b_test))

# create one hot vector from the train labels
train_out_int = np.array([int(i) for i in train_out])
y_true = np.zeros((train_out_int.size, train_out_int.max() + 1))
y_true[np.arange(train_out_int.size), train_out_int] = 1 # shape: (1707, 10)

# create one hot vector from the test labels
test_out_int = np.array([int(i) for i in test_out])
y_true_test = np.zeros((test_out_int.size, test_out_int.max() + 1))
y_true_test[np.arange(test_out_int.size), test_out_int] = 1 # shape: (1707, 10)

epochs = 100

In [8]:
accuracies_per_init = {}
inits = {"random": W_random, "ones": W_ones, "zeros": W_zeros}

for name, W in inits.items():
    accuracies = []
    for e in range(epochs):
        # Forward pass
        scores = T @ W
        preds = np.argmax(scores, axis=1)
        y_pred = np.zeros((preds.size, nr_digits))
        y_pred[np.arange(preds.size), preds] = 1
        
        # forward pass on test data for accuracy computation
        test_scores = T_test @ W
        test_preds = np.argmax(test_scores, axis=1)
        y_test_pred = np.zeros((test_preds.size, nr_digits))
        y_test_pred[np.arange(test_preds.size), test_preds] = 1
        
        
        # Compute loss
        Error = (y_true - y_pred)
        
        # Compute gradients
        grads = T.T @ Error
        
        # gradient descent step
        W = W + lr * grads
        
        accuracy = np.mean(y_test_pred == y_true_test)
        accuracies.append(accuracy)
        # print(f"Accuracy: {accuracy} at Epoch: {e}")
    
    accuracies_per_init[name] = accuracies

for name, accuracies in accuracies_per_init.items():
    print(f"Init Strat: {name}, Final accuracy: {accuracies[-1]} after {epochs} epochs")

Init Strat: random, Final accuracy: 0.975 after 100 epochs
Init Strat: ones, Final accuracy: 0.9744 after 100 epochs
Init Strat: zeros, Final accuracy: 0.9744 after 100 epochs
